## 만들었던 데이터 합치기
___
### A. 합치기 전에 파일 확인해야 할 것
- 시군구 이름이 다 정확하게 있는지 확인하기<br>
- 결측치 있는지 확인하기(blank나 "-")<br>
- 시군구 이름 오름차순으로 정렬해서 같은 위치에 저장 덮어쓰기<br>
---
### B. 만들어야 할 것
1. cofog별로 파일 합치기<br>
  1-1. 시군구명 적용된 파일<br>
  1-2. 시군구코드 적용된 파일

2. 모든 cofog 파일 합치기<br>
  2-1. 시군구명 적용된 파일<br>
  2-2. 시군구코드 적용된 파일
---
### C. 폴더 구조, 파일명 예시
1. cofog별<br>
  - 시군구명<br>
    - 예: `공공질서및안전_시군구이름.csv`<br>
  - 시군구코드<br>
    - 예: `공공질서및안전.csv`<br>

2. 전체<br>
  - 시군구명<br>
    - 예: `data_시군구이름.csv`<br>
  - 시군구코드<br>
    - 예: `data.csv`<br>

In [ ]:
import os
import glob
import pandas as pd

### A. 합치기 전에 파일 확인해야 할 것
- 시군구 이름이 다 정확하게 있는지 확인하기<br>
- 결측치 있는지 확인하기(blank나 "-")<br>
- 시군구 이름 오름차순으로 정렬해서 같은 위치에 저장 덮어쓰기<br>

In [ ]:
region_list = [
    "강원특별자치도 강릉시", "강원특별자치도 고성군", "강원특별자치도 동해시", "강원특별자치도 삼척시", "강원특별자치도 속초시",
    "강원특별자치도 양구군", "강원특별자치도 양양군", "강원특별자치도 영월군", "강원특별자치도 원주시", "강원특별자치도 인제군",
    "강원특별자치도 정선군", "강원특별자치도 철원군", "강원특별자치도 춘천시", "강원특별자치도 태백시", "강원특별자치도 평창군",
    "강원특별자치도 홍천군", "강원특별자치도 화천군", "강원특별자치도 횡성군", "경상남도 거제시", "경상남도 거창군",
    "경상남도 고성군", "경상남도 김해시", "경상남도 남해군", "경상남도 밀양시", "경상남도 사천시", "경상남도 산청군",
    "경상남도 양산시", "경상남도 의령군", "경상남도 진주시", "경상남도 창녕군", "경상남도 창원시", "경상남도 통영시",
    "경상남도 하동군", "경상남도 함안군", "경상남도 함양군", "경상남도 합천군", "경상북도 경산시", "경상북도 경주시",
    "경상북도 고령군", "경상북도 구미시", "경상북도 김천시", "경상북도 문경시", "경상북도 봉화군", "경상북도 상주시",
    "경상북도 성주군", "경상북도 안동시", "경상북도 영덕군", "경상북도 영양군", "경상북도 영주시", "경상북도 영천시",
    "경상북도 예천군", "경상북도 울릉군", "경상북도 울진군", "경상북도 의성군", "경상북도 청도군", "경상북도 청송군",
    "경상북도 칠곡군", "경상북도 포항시", "광주광역시 광산구", "광주광역시 남구", "광주광역시 동구", "광주광역시 북구",
    "광주광역시 서구", "대구광역시 군위군", "대구광역시 남구", "대구광역시 달서구", "대구광역시 달성군", "대구광역시 동구",
    "대구광역시 북구", "대구광역시 서구", "대구광역시 수성구", "대구광역시 중구", "대전광역시 대덕구", "대전광역시 동구",
    "대전광역시 서구", "대전광역시 유성구", "대전광역시 중구", "부산광역시 강서구", "부산광역시 금정구", "부산광역시 기장군",
    "부산광역시 남구", "부산광역시 동구", "부산광역시 동래구", "부산광역시 북구", "부산광역시 사상구", "부산광역시 사하구",
    "부산광역시 서구", "부산광역시 수영구", "부산광역시 연제구", "부산광역시 영도구", "부산광역시 중구", "부산광역시 부산진구",
    "부산광역시 해운대구", "세종특별자치시", "울산광역시 남구", "울산광역시 동구", "울산광역시 북구", "울산광역시 울주군",
    "울산광역시 중구", "전라남도 강진군", "전라남도 고흥군", "전라남도 곡성군", "전라남도 광양시", "전라남도 구례군",
    "전라남도 나주시", "전라남도 담양군", "전라남도 목포시", "전라남도 무안군", "전라남도 보성군", "전라남도 순천시",
    "전라남도 신안군", "전라남도 여수시", "전라남도 영광군", "전라남도 영암군", "전라남도 완도군", "전라남도 장성군",
    "전라남도 장흥군", "전라남도 진도군", "전라남도 함평군", "전라남도 해남군", "전라남도 화순군", "전북특별자치도 고창군",
    "전북특별자치도 군산시", "전북특별자치도 김제시", "전북특별자치도 남원시", "전북특별자치도 무주군", "전북특별자치도 부안군",
    "전북특별자치도 순창군", "전북특별자치도 완주군", "전북특별자치도 익산시", "전북특별자치도 임실군", "전북특별자치도 장수군",
    "전북특별자치도 전주시", "전북특별자치도 정읍시", "전북특별자치도 진안군", "제주특별자치도 서귀포시", "제주특별자치도 제주시",
    "충청남도 계룡시", "충청남도 공주시", "충청남도 금산군", "충청남도 논산시", "충청남도 당진시", "충청남도 보령시",
    "충청남도 부여군", "충청남도 서산시", "충청남도 서천군", "충청남도 아산시", "충청남도 예산군", "충청남도 천안시",
    "충청남도 청양군", "충청남도 태안군", "충청남도 홍성군", "충청북도 괴산군", "충청북도 단양군", "충청북도 보은군",
    "충청북도 영동군", "충청북도 옥천군", "충청북도 음성군", "충청북도 제천시", "충청북도 증평군", "충청북도 진천군",
    "충청북도 청주시", "충청북도 충주시",
    "서울특별시 종로구", "서울특별시 중구", "서울특별시 용산구", "서울특별시 성동구",
    "서울특별시 광진구", "서울특별시 동대문구", "서울특별시 중랑구", "서울특별시 성북구",
    "서울특별시 강북구", "서울특별시 도봉구", "서울특별시 노원구", "서울특별시 은평구",
    "서울특별시 서대문구", "서울특별시 마포구", "서울특별시 양천구", "서울특별시 강서구",
    "서울특별시 구로구", "서울특별시 금천구", "서울특별시 영등포구", "서울특별시 동작구",
    "서울특별시 관악구", "서울특별시 서초구", "서울특별시 강남구", "서울특별시 송파구",
    "서울특별시 강동구",
    "경기도 가평군", "경기도 고양시", "경기도 과천시", "경기도 광명시", "경기도 광주시", 
    "경기도 구리시", "경기도 군포시", "경기도 김포시", "경기도 남양주시", "경기도 동두천시", 
    "경기도 부천시", "경기도 성남시", "경기도 수원시", "경기도 시흥시", "경기도 안산시",
    "경기도 안성시", "경기도 안양시", "경기도 양주시", "경기도 양평군", "경기도 여주시", 
    "경기도 연천군", "경기도 오산시", "경기도 용인시", "경기도 의왕시", "경기도 의정부시", 
    "경기도 이천시", "경기도 파주시", "경기도 평택시", "경기도 포천시", "경기도 하남시", 
    "경기도 화성시",
    "인천광역시 강화군", "인천광역시 옹진군", "인천광역시 계양구", "인천광역시 미추홀구",
    "인천광역시 남동구", "인천광역시 동구", "인천광역시 부평구", "인천광역시 서구",
    "인천광역시 연수구", "인천광역시 중구"
]

In [ ]:
# 시군구 이름이 다 정확하게 있는지 확인하기
def load_csv_from_folder(folder_path):
    """
    지정된 폴더 내 모든 CSV 파일을 읽어서 DataFrame 목록으로 반환
    - utf-8-sig → euc-kr 순서로 인코딩 시도
    """
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    dataframes = {}

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        try:
            # 1차 시도: UTF-8-SIG
            df = pd.read_csv(file_path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            try:
                # 2차 시도: EUC-KR
                df = pd.read_csv(file_path, encoding="euc-kr")
            except Exception as e:
                print(f"❌ {file_name} 읽기 실패: {e}")
                continue
        except Exception as e:
            print(f"❌ {file_name} 읽기 실패: {e}")
            continue
        
        # UTF-8-SIG로 형식 변경하여 저장
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        dataframes[file_name] = df
    
    return dataframes

def check_region_in_df(df, region_col="SGG_NAME"):
    """
    DataFrame에서 시군구 컬럼이 region_list와 일치하는지 확인
    """
    unique_regions = sorted(df[region_col].dropna().unique())
    set_file = set(unique_regions)
    set_master = set(region_list)
    
    missing_in_file = sorted(set_master - set_file)  # 있어야 하는데 없는 시군구
    extra_in_file = sorted(set_file - set_master)    # region_list에 없는데 파일에 있는 시군구
    
    return {
        "unique_count": len(unique_regions),
        "match_229": len(unique_regions) == len(region_list),
        "missing_count": len(missing_in_file),
        "extra_count": len(extra_in_file),
        "missing_list": missing_in_file,
        "extra_list": extra_in_file
    }

def validate_df(dfs, folder_path):
    for name, df in dfs.items():
        print(folder_path, name)
        if "SGG_NAME" not in df.columns:
            print(f"⚠️ {name} → SGG_NAME 컬럼 없음")
            continue
        result = check_region_in_df(df)
        print(f"📄 {name}")
        print(f" - 고유 시군구 개수: {result['unique_count']} / 기준 229개 일치 여부: {result['match_229']}")
        print(f" - 빠진 시군구 {result['missing_count']}개 / 추가 시군구 {result['extra_count']}개")
        if result["missing_count"] > 0:
            print(f"   빠진 시군구: {result['missing_list']}")
        if result["extra_count"] > 0:
            print(f"   추가 시군구: {result['extra_list']}")
        df = df.sort_values(by="SGG_NAME", ascending=True)
        df.to_csv(f"{folder_path}/{name}", index=False, encoding="utf-8-sig")
        print()

base_path = "../../data/01-3_merge/병합전"
folder_path_list = []

for root, dirs, files in os.walk(base_path):
    for d in dirs:
        folder_path_list.append(os.path.join(root, d))

# print(folder_path_list)

for folder_path in folder_path_list:
    dfs = load_csv_from_folder(folder_path)
    validate_df(dfs, folder_path)

### B. 만들어야 할 것
1. cofog별로 파일 합치기<br>
  1-1. 시군구명 적용된 파일<br>
  1-2. 시군구코드 적용된 파일

In [ ]:
import os
import glob
from functools import reduce

BASE_DIR = "../../data/01-3_merge/병합전"             # 상위 폴더
SAVE_DIR = "../../data/01-3_merge/병합후/cofog별/시군구이름"     # 저장 폴더
NAME_KEY = "SGG_NAME"

os.makedirs(SAVE_DIR, exist_ok=True)  # 저장 폴더 없으면 생성

def read_csv_auto(path):
    for enc in ["utf-8-sig", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] == 1:  # 탭 구분자 처리
                df = pd.read_csv(path, encoding=enc, sep="\t")
            return df
        except:
            continue
    return None

def merge_by_key(folder_path, key):
    csvs = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = []
    for p in csvs:
        df = read_csv_auto(p)
        if df is not None and key in df.columns:
            dfs.append(df)
    if not dfs:
        return None
    return reduce(lambda l, r: pd.merge(l, r, on=key, how="outer"), dfs)

# 실행
subfolders = [os.path.join(BASE_DIR, f) for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))]
for folder in subfolders:
    name = os.path.basename(folder)
    
    # 1-1 SGG_NAME 기준
    merged_name = merge_by_key(folder, NAME_KEY)
    if merged_name is not None:
        merged_name.to_csv(os.path.join(SAVE_DIR, f"{name}_시군구이름.csv"), index=False, encoding="utf-8-sig")
        print(f"✅ {name} 이름 기준 병합 완료")
    else:
        print(f"⚠️ {name} 이름 기준 병합 불가")


In [ ]:
import os
import glob
import pandas as pd

# === 매핑 파일 불러오기 ===
mapping_path = "../../data/01-3_merge/SGG_CODE.csv"  # 매핑 CSV 경로
mapping_df = pd.read_csv(mapping_path, encoding="utf-8-sig")
mapping_df["SGG_CODE"] = mapping_df["SGG_CODE"].astype(str).str.strip()
mapping_df["SGG_NAME"] = mapping_df["SGG_NAME"].str.strip()

def read_csv_auto(path):
    for enc in ["utf-8-sig", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] == 1:  # 탭 구분자 처리
                df = pd.read_csv(path, encoding=enc, sep="\t")
            return df
        except:
            continue
    return None

def apply_code_mapping_and_reorder(src_folder, save_folder):
    os.makedirs(save_folder, exist_ok=True)

    csv_files = glob.glob(os.path.join(src_folder, "*.csv"))
    for file_path in csv_files:
        df = read_csv_auto(file_path)
        if df is None:
            continue
        
        # SGG_NAME만 있는 경우
        if "SGG_NAME" in df.columns and "SGG_CODE" not in df.columns:
            df["SGG_NAME"] = df["SGG_NAME"].str.strip()
            df = pd.merge(df, mapping_df, on="SGG_NAME", how="left")

            # SGG_NAME 삭제
            # df.drop(columns=["SGG_NAME"], inplace=True)

            # SGG_CODE 제일 왼쪽으로 이동
            cols = ["SGG_CODE"] + [c for c in df.columns if c != "SGG_CODE"]
            df = df[cols]

            # SGG_CODE 기준 정렬
            df["SGG_CODE"] = df["SGG_CODE"].astype(str)
            df = df.sort_values(by="SGG_CODE").reset_index(drop=True)

            print(f"✅ {os.path.basename(file_path)} → SGG_CODE 추가/정렬 완료")
        else:
            print(f"ℹ️ {os.path.basename(file_path)} → 변환 불필요 또는 이미 SGG_CODE 있음")

        # 저장 파일명에서 "_시군구이름" 제거 + 확장자 제거 + "_전체.csv" 추가
        base_name = os.path.basename(file_path).replace("_시군구이름", "")
        base_name_no_ext = os.path.splitext(base_name)[0]  # 확장자 제거
        save_path = os.path.join(save_folder, base_name_no_ext + "_전체.csv")

        # 저장
        df.to_csv(save_path, index=False, encoding="utf-8-sig")

# === 실행 ===
src_folder = "../../data/01-3_merge/병합후/cofog별/시군구이름"             # 원본 CSV 폴더
save_folder = "../../data/01-3_merge/병합후/cofog별/전체"  # 결과 저장 폴더

apply_code_mapping_and_reorder(src_folder, save_folder)


In [ ]:
## 전체 병합

import os
import glob
from functools import reduce

BASE_DIR = "../../data/01-3_merge/병합후/cofog별/시군구이름"             # 상위 폴더
SAVE_DIR = "../../data/01-3_merge/병합후/전체"     # 저장 폴더
NAME_KEY = "SGG_NAME"
CODE_KEY = "SGG_CODE"

os.makedirs(SAVE_DIR, exist_ok=True)  # 저장 폴더 없으면 생성

def read_csv_auto(path):
    for enc in ["utf-8-sig", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] == 1:  # 탭 구분자 처리
                df = pd.read_csv(path, encoding=enc, sep="\t")
            return df
        except:
            continue
    return None

def merge_by_key(folder_path, key):
    csvs = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = []
    for p in csvs:
        df = read_csv_auto(p)
        if df is not None and key in df.columns:
            dfs.append(df)
    if not dfs:
        return None
    return reduce(lambda l, r: pd.merge(l, r, on=key, how="outer"), dfs)

# 실행
subfolders = [os.path.join(BASE_DIR, f) for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))]
for folder in subfolders:
    name = os.path.basename(folder)
    
    # 1-1 SGG_NAME 기준
    merged_name = merge_by_key(folder, NAME_KEY)
    if merged_name is not None:
        merged_name.to_csv(os.path.join(SAVE_DIR, f"data_시군구이름.csv"), index=False, encoding="utf-8-sig")
        print(f"✅ {name} 이름 기준 병합 완료")
    else:
        print(f"⚠️ {name} 이름 기준 병합 불가")

    # 1-2 SGG_CODE 기준
    merged_code = merge_by_key(folder, CODE_KEY)
    if merged_code is not None:
        merged_code.to_csv(os.path.join(SAVE_DIR, f"data.csv"), index=False, encoding="utf-8-sig")
        print(f"✅ {name} 코드 기준 병합 완료")
    else:
        print(f"⚠️ {name} 코드 기준 병합 불가")

In [ ]:
import os
import glob
import pandas as pd
from functools import reduce

FOLDER_PATH = "../../data/01-3_merge/병합전/cofog별/시군구코드"  # CSV들이 들어 있는 폴더
SAVE_PATH = "../../data/01-3_merge/병합전/전체/data.csv"  # 저장 경로
KEY_COL = "SGG_CODE"

def read_csv_auto(path):
    for enc in ["utf-8-sig", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] == 1:  # 탭 구분자 처리
                df = pd.read_csv(path, encoding=enc, sep="\t")
            return df
        except:
            continue
    return None

def merge_all_by_key(folder_path, key):
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = []
    for p in csv_files:
        df = read_csv_auto(p)
        if df is not None and key in df.columns:
            dfs.append(df)
        else:
            print(f"⚠️ {os.path.basename(p)} → {key} 컬럼 없음, 건너뜀")
    if not dfs:
        return None
    return reduce(lambda l, r: pd.merge(l, r, on=key, how="outer"), dfs)

# 실행
merged_df = merge_all_by_key(FOLDER_PATH, KEY_COL)

if merged_df is not None:
    merged_df = merged_df.sort_values(by=KEY_COL).reset_index(drop=True)  # 시군구 오름차순 정렬
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    merged_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ 병합 완료 → {SAVE_PATH} ({merged_df.shape})")
else:
    print("⚠️ 병합할 데이터가 없습니다.")


In [ ]:
import os
import glob
import pandas as pd
from functools import reduce

FOLDER_PATH = "../../data/01-3_merge/병합후/cofog별/전체"  # CSV들이 들어 있는 폴더
SAVE_PATH = "../../data/01-3_merge/병합후/전체/data_전체.csv"  # 저장 경로
KEY_COLS = ["SGG_CODE", "SGG_NAME"]  # 병합 기준 컬럼

def read_csv_auto(path):
    for enc in ["utf-8-sig", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] == 1:  # 탭 구분자 처리
                df = pd.read_csv(path, encoding=enc, sep="\t")
            return df
        except:
            continue
    return None

def merge_all_by_keys(folder_path, keys):
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = []
    for p in csv_files:
        df = read_csv_auto(p)
        if df is not None and all(k in df.columns for k in keys):
            dfs.append(df)
        else:
            print(f"⚠️ {os.path.basename(p)} → {keys} 중 일부 없음, 건너뜀")
    if not dfs:
        return None
    return reduce(lambda l, r: pd.merge(l, r, on=keys, how="outer"), dfs)

# 실행
merged_df = merge_all_by_keys(FOLDER_PATH, KEY_COLS)

if merged_df is not None:
    merged_df = merged_df.sort_values(by=KEY_COLS).reset_index(drop=True)  # 시군구코드, 시군구명 순으로 정렬
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    merged_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")
    print(f"✅ 병합 완료 → {SAVE_PATH} ({merged_df.shape})")
else:
    print("⚠️ 병합할 데이터가 없습니다.")

# 잘됬는지 확인

In [ ]:
import os
import glob
import pandas as pd

BASE_DIR = "../../data/01-3_merge/병합후"  # 검사할 상위 폴더
EXPECTED_ROWS = 229

def read_csv_auto(path):
    """UTF-8-SIG → EUC-KR 순서로 읽고, 탭 구분 처리"""
    for enc in ["utf-8-sig", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] == 1:
                df = pd.read_csv(path, encoding=enc, sep="\t")
            return df
        except:
            continue
    return None

def check_files(base_dir):
    csv_files = glob.glob(os.path.join(base_dir, "**", "*.csv"), recursive=True)
    results = []

    for file_path in csv_files:
        df = read_csv_auto(file_path)
        if df is None:
            results.append({
                "file": file_path,
                "rows": None,
                "rows_ok": False,
                "missing_values": None
            })
            continue
        
        # 행 개수 확인
        rows = len(df)
        rows_ok = (rows == EXPECTED_ROWS)

        # 결측치 여부 확인
        missing_values = df.isna().any().any()

        results.append({
            "file": file_path,
            "rows": rows,
            "rows_ok": rows_ok,
            "missing_values": missing_values
        })

    return pd.DataFrame(results)

# 실행
df_check = check_files(BASE_DIR)

# 보기 좋게 표시
df_check = df_check.sort_values(by="file").reset_index(drop=True)
print(df_check)

# 저장 (선택)
df_check.to_csv("../../data/01-3_merge/병합후/파일-검사결과.csv", index=False, encoding="utf-8-sig")
print("✅ 검사 완료: ../../data/01-3_merge/병합후/파일-검사결과.csv 저장됨")
